# Week 15 실습 · 최종 프로젝트 발표와 재현성

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MNU-AI-Programming/AI-Assisted-Python-Programming/blob/main/weeks/week15/practice.ipynb)

**AI활용코딩연습(ZBE135)** · 이영호 교수 · 국립목포대학교 컴퓨터학부

## 오늘의 목표

1. **재현성**(다른 사람이 같은 결과를 얻는 것)을 깨뜨리는 원인을 알고 막습니다.
2. 환경·데이터·난수 시드를 기록하는 **재현성 셀**을 노트북 맨 위에 둡니다.
3. **질문 → 데이터 → 정제 → 분석 → 시각화 → 결론·한계 → AI 활용** 구조의 최종 노트북을 완성합니다.
4. README와 **5분 발표**를 준비합니다.
5. AI가 쓴 결론 문장을 근거로 검증하고, **AI 활용을 정직하게 공개**합니다.

| 구간 | 내용 | 시간 |
|---|---|---|
| 1 | 재현성이 깨지는 이유 | 10분 |
| 2 | 재현성 셀 만들기 | 15분 |
| 3 | 예시 미니 프로젝트 (처음부터 끝까지) | 30분 |
| 4 | README와 5분 발표 | 25분 |
| 5 | AI 결론 검증과 AI 활용 공개 | 15분 |
| 6 | 최종 점검 · 정리 | 25분 |

> 수업 운영: 발표 주간에는 앞 50분을 이 노트북으로 최종 점검하고, 나머지 시간에 **5분 발표 + 2분 질의**를 진행합니다.

---
## 1. 재현성이 깨지는 이유

| 원인 | 증상 | 막는 법 |
|---|---|---|
| 셀 실행 순서 | 내 화면에서만 동작 | **런타임 → 다시 시작 및 모두 실행** |
| 절대 경로 | `/content/drive/MyDrive/내파일.csv` → 남의 컴퓨터에 없음 | 상대 경로 또는 공개 URL (`get_data`) |
| 난수 | 실행할 때마다 결과가 바뀜 | `np.random.default_rng(42)`, `random_state=42` |
| 라이브러리 버전 | 함수 이름·기본값이 달라짐 | 버전 기록, `requirements.txt` |
| 데이터가 바뀜 | 같은 코드인데 숫자가 다름 | 데이터 **지문**(행 수·해시) 기록, 출처·날짜 |
| 손으로 고친 데이터 | 과정이 기록에 없음 | 정제도 **코드로** |

---
## 2. 재현성 셀 만들기

최종 노트북의 **맨 위**에 다음 셀을 둡니다. 결과가 달라졌을 때 원인을 찾는 단서가 됩니다.

In [ ]:
import sys
import hashlib
import platform
from datetime import datetime
from pathlib import Path
import urllib.request

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

try:
    import koreanize_matplotlib
except ModuleNotFoundError:
    %pip install -q koreanize-matplotlib
    import koreanize_matplotlib

SEED = 42
rng = np.random.default_rng(SEED)

DATA_URL = "https://raw.githubusercontent.com/MNU-AI-Programming/AI-Assisted-Python-Programming/main/datasets/"

def get_data(name):
    for folder in [Path("../../datasets"), Path("datasets"), Path(".")]:
        if (folder / name).exists():
            return str(folder / name)
    urllib.request.urlretrieve(DATA_URL + name, name)
    return name


def fingerprint(path):
    """데이터 파일의 지문: 행 수와 MD5 해시 앞 12자리."""
    raw = Path(path).read_bytes()
    return {"file": Path(path).name, "rows": raw.count(b"\n") - 1, "md5": hashlib.md5(raw).hexdigest()[:12]}


print(f"실행 시각  : {datetime.now():%Y-%m-%d %H:%M}")
print(f"Python     : {sys.version.split()[0]} ({platform.system()})")
print(f"numpy {np.__version__} / pandas {pd.__version__} / matplotlib {matplotlib.__version__}")
print(f"난수 시드  : {SEED}")
for name in ["cafe_sales.csv", "weather_daily.csv"]:
    print("데이터     :", fingerprint(get_data(name)))

### 난수와 재현성

In [ ]:
values = np.arange(1, 101)

print("시드 없음:", [np.random.default_rng().choice(values, 3).tolist() for _ in range(2)])
print("시드 고정:", [np.random.default_rng(SEED).choice(values, 3).tolist() for _ in range(2)])

df_demo = pd.DataFrame({"x": values})
print("sample random_state:", df_demo.sample(3, random_state=SEED)["x"].tolist(),
      df_demo.sample(3, random_state=SEED)["x"].tolist())

---
## 3. 예시 미니 프로젝트: 처음부터 끝까지

최종 노트북이 어떤 모습이어야 하는지 **짧은 예시**로 보여 줍니다. 각 절의 제목과 순서를 그대로 따라 하세요.

### 3-1. 질문

1. **평일 점심(11~13시)** 은 하루 매출의 몇 %를 차지하는가?
2. **기온이 20℃ 이상인 날**은 10℃ 미만인 날보다 아이스 음료 비율이 얼마나 높은가?

### 3-2. 데이터

| 데이터 | 출처 | 한 행 | 기간 |
|---|---|---|---|
| `cafe_sales.csv` | 수업용 가상 데이터 | 주문 한 건의 메뉴 한 종류 | 2025-03-03 ~ 06-20 (일요일 휴무) |
| `weather_daily.csv` | 수업용 가상 데이터 | 하루 | 2025년 1년 |

In [ ]:
cafe = pd.read_csv(get_data("cafe_sales.csv"), parse_dates=["date"])
weather = pd.read_csv(get_data("weather_daily.csv"), parse_dates=["date"])
print(cafe.shape, weather.shape, cafe["date"].min().date(), cafe["date"].max().date())

### 3-3. 정제·파생 변수 (함수로)

In [ ]:
def prepare(cafe):
    """분석용 열 추가 (입력 불변)."""
    hour = cafe["time"].str[:2].astype(int)
    return cafe.assign(
        revenue=cafe["unit_price"] * cafe["qty"],
        hour=hour,
        weekday=cafe["date"].dt.dayofweek,          # 0=월 … 5=토
        lunch=hour.between(11, 13),
    )


def daily_table(cafe, weather):
    """일별 매출·점심 매출·아이스 비율·기온."""
    c = prepare(cafe)
    drinks = c[c["temp_type"].isin(["HOT", "ICE"])]
    ice_qty = drinks["qty"].where(drinks["temp_type"] == "ICE", 0).groupby(drinks["date"]).sum()
    day = c.groupby("date").agg(revenue=("revenue", "sum"), weekday=("weekday", "first"))
    day["lunch_revenue"] = c[c["lunch"]].groupby("date")["revenue"].sum()
    day["ice_pct"] = ice_qty / drinks.groupby("date")["qty"].sum() * 100
    out = day.reset_index().merge(weather[["date", "avg_temp"]], on="date", how="left", validate="one_to_one")
    assert len(out) == c["date"].nunique()
    return out


day = daily_table(cafe, weather)
day.head()

### 3-4. 분석

In [ ]:
weekday_days = day[day["weekday"] < 5]
lunch_share = weekday_days["lunch_revenue"].sum() / weekday_days["revenue"].sum() * 100

warm = day.loc[day["avg_temp"] >= 20, "ice_pct"]
cold = day.loc[day["avg_temp"] < 10, "ice_pct"]

summary = pd.DataFrame({
    "질문": ["평일 점심 매출 비중", "아이스 비율: 20℃ 이상", "아이스 비율: 10℃ 미만"],
    "값": [f"{lunch_share:.1f}%", f"{warm.mean():.1f}%", f"{cold.mean():.1f}%"],
    "근거(일수)": [len(weekday_days), len(warm), len(cold)],
})
summary

### 3-5. 작은 테스트 (핵심 계산 확인)

In [ ]:
def test_daily_table_small():
    c = pd.DataFrame({"order_id": ["1", "2", "3"], "date": pd.to_datetime(["2025-03-03"] * 3),
                      "time": ["12:10", "09:00", "12:30"], "menu": ["A", "B", "C"], "category": ["커피", "커피", "디저트"],
                      "temp_type": ["ICE", "HOT", "-"], "unit_price": [1000, 2000, 3000], "qty": [3, 1, 1], "payment": ["카드"] * 3})
    w = pd.DataFrame({"date": pd.to_datetime(["2025-03-03"]), "avg_temp": [15.0]})
    d = daily_table(c, w)
    assert d.loc[0, "revenue"] == 8000
    assert d.loc[0, "lunch_revenue"] == 6000          # 12:10, 12:30
    assert d.loc[0, "ice_pct"] == 75.0                # ICE 3 / (3+1)
    assert len(c.columns) == 9                        # 입력 불변


test_daily_table_small()
print("핵심 계산 테스트 통과 ✅")

### 3-6. 시각화와 저장

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

hourly = prepare(cafe).query("weekday < 5").groupby("hour")["revenue"].sum() / weekday_days["date"].nunique() / 1000
colors = ["#C0392B" if 11 <= h <= 13 else "#BBBBBB" for h in hourly.index]
axes[0].bar(hourly.index, hourly.values, color=colors)
axes[0].set_title(f"평일 점심(11~13시)이 매출의 {lunch_share:.0f}%", loc="left", fontweight="bold")
axes[0].set_xlabel("시각 (시)")
axes[0].set_ylabel("일평균 매출 (천 원)")

bands = pd.cut(day["avg_temp"], [-np.inf, 10, 15, 20, 25, np.inf], labels=["<10", "10~15", "15~20", "20~25", "25≤"])
by_band = day.groupby(bands, observed=True)["ice_pct"].mean()
axes[1].bar(by_band.index.astype(str), by_band.values, color="#4C78A8")
axes[1].set_title("기온이 높은 날일수록 아이스 비율이 높다", loc="left", fontweight="bold")
axes[1].set_xlabel("평균기온 구간 (℃)")
axes[1].set_ylabel("아이스 비율 (%)")
axes[1].set_ylim(0, 100)

for a in axes:
    a.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
Path("figures").mkdir(exist_ok=True)
fig.savefig("figures/final_example.png", dpi=150, bbox_inches="tight")
plt.show()

### 3-7. 결론과 한계

**결론**

1. 평일에는 점심 3시간(11~13시)이 하루 매출의 **39.2%** 를 차지한다. → 점심 시간 인력 배치가 중요하다.
2. 평균기온 20℃ 이상인 날의 아이스 음료 비율(74.2%)은 10℃ 미만인 날(26.0%)보다 **약 48%p 높다**.

**한계**

- 가상 데이터이며, 한 학기(약 4개월)만 포함한다.
- 기온과 아이스 비율은 **함께 변한다**는 것이지, 기온이 원인이라는 증거는 아니다 (계절·학사 일정이 함께 바뀜).
- 25℃ 이상인 날은 6일뿐이라 해당 구간 값은 불안정하다.

**AI 활용**: 13주차 대시보드 코드의 초안을 AI로 만들고, 요일 순서·y축 0 시작·일평균 계산을 직접 고쳤다. 결론 문장은 AI 초안에서 "기온이 매출을 올린다"는 인과 표현을 삭제했다.

---
## 4. README와 5분 발표

### README 구성

GitHub나 제출 폴더의 첫 화면입니다. **처음 보는 사람이 5분 안에** 무엇을 했고, 어떻게 다시 실행하는지 알 수 있어야 합니다.

In [ ]:
readme = """# 캠퍼스 카페 판매 패턴 분석

> 한 줄 요약: 평일 점심이 매출의 약 40%를 차지하고, 기온이 높은 날 아이스 음료 비율이 크게 높다.

## 연구 질문
1. 평일 점심(11~13시)은 하루 매출의 몇 %를 차지하는가?
2. 기온 20℃ 이상인 날은 10℃ 미만인 날보다 아이스 음료 비율이 얼마나 높은가?

## 데이터
| 파일 | 출처 | 기간 | 비고 |
|---|---|---|---|
| cafe_sales.csv | 수업용 가상 데이터 | 2025-03-03 ~ 06-20 | 3,019행 |
| weather_daily.csv | 수업용 가상 데이터 | 2025년 | 365행 |

## 실행 방법
1. Colab에서 final_project.ipynb를 연다.
2. 런타임 → 다시 시작 및 모두 실행 (데이터는 GitHub에서 자동으로 내려받음)
3. 그림은 figures/ 폴더에 저장된다.

## 주요 결과
![결과](figures/final_example.png)

## 한계
- 가상 데이터, 한 학기 기간, 상관관계(인과 아님)

## AI 활용
- 사용 도구와 범위, 직접 검증·수정한 내용은 ai_use_log.md 참고

## 파일 구성
- final_project.ipynb, README.md, figures/, ai_use_log.md, slides.pdf
"""
Path("README_example.md").write_text(readme, encoding="utf-8")
print(readme[:300], "...")

### 5분 발표 구성 (슬라이드 6~8장)

| 순서 | 내용 | 시간 |
|---|---|---:|
| 1 | 문제와 연구 질문 (왜 궁금한가?) | 40초 |
| 2 | 데이터: 출처, 한 행의 의미, 기간 | 30초 |
| 3 | 방법: 정제·비교 방법 한 장 요약 | 40초 |
| 4~5 | **결과** 2~3개: 결론형 제목 + 그래프 하나씩 | 2분 |
| 6 | 한계와 다음 단계 | 30초 |
| 7 | AI를 어떻게 쓰고 검증했나 | 30초 |
| | 질의응답 | 2분 |

**발표 팁**

- 그래프 한 장에 **메시지 하나**. 제목이 곧 말할 문장입니다.
- 숫자는 **비교와 함께**: "40%" 보다 "점심 3시간이 하루의 40%"
- 코드 화면은 꼭 필요할 때만, 짧게
- 리허설로 **시간을 재고**, 4분 30초에 끝나도록

---
## 5. AI 결론 검증과 AI 활용 공개

AI에게 결과표를 주고 결론을 써 달라고 하면 이런 문장이 나오곤 합니다. 각 문장을 **근거와 대조**해 봅시다.

| AI 문장 | 근거 확인 | 판정 |
|---|---|---|
| "점심 시간 매출이 압도적으로 높아 하루 매출의 절반 이상을 차지합니다." | 평일 점심 비중 약 40% | ❌ 과장 |
| "기온 상승이 아이스 음료 수요를 증가시킵니다." | 상관관계만 확인 | ❌ 인과 주장 |
| "25℃ 이상인 날 아이스 비율이 가장 높았습니다(6일 기준)." | 구간별 표 | ✅ 근거·표본 명시 |

### AI 활용 공개 문장 쓰기

좋은 공개는 **무엇을, 어디까지, 어떻게 검증했는지**가 구체적입니다.

| ❌ | ✅ |
|---|---|
| "AI를 참고했습니다." | "그래프 초안 코드(13주차 대시보드)를 ChatGPT로 생성했고, 요일 순서·y축 시작점·일평균 계산을 직접 수정했다. 결론 문장의 인과 표현은 삭제했다." |
| "AI 사용 안 함" (실제로는 사용) | 사용한 만큼 정직하게 — 사용 자체는 감점 대상이 아니다 |

---
## 6. 최종 점검 · 정리

### 제출 전 체크리스트

**재현성**
- [ ] 맨 위에 재현성 셀(버전·시드·데이터 지문)이 있다
- [ ] **런타임 → 다시 시작 및 모두 실행**에서 오류 없이 끝까지 실행된다
- [ ] 개인 드라이브 절대 경로가 없다 (상대 경로나 공개 URL)

**내용**
- [ ] 질문 → 데이터 → 정제 → 분석 → 시각화 → 결론·한계 → AI 활용 순서
- [ ] 핵심 계산 함수에 작은 테스트가 있다
- [ ] 그래프마다 결론형 제목과 단위가 있다
- [ ] 결론마다 근거(숫자·표·그래프)와 한계가 있다

**제출물**
- [ ] `final_project.ipynb`, `README.md`, 발표 자료(PDF), `ai_use_log.md`, 성찰 보고서

In [ ]:
# 실습 중 만든 파일 정리
for p in ["README_example.md", "figures/final_example.png"]:
    Path(p).unlink(missing_ok=True)
print("정리 완료")

### 한 학기 정리

| 구간 | 배운 것 | 한 문장 |
|---|---|---|
| 1~5주 | Python 기초와 함수·테스트 | AI 코드를 **실행하고 검증**한다 |
| 6~8주 | 파일·예외·클래스, 중간 실기 | 데이터와 규칙을 **견고하게** 다룬다 |
| 9~13주 | NumPy·pandas·정제·EDA·시각화 | 질문에서 출발해 **근거 있는 결론**을 낸다 |
| 14~15주 | 디버깅·테스트·리팩토링·재현성 | 남이 **다시 실행할 수 있는** 결과를 만든다 |

> AI는 빠른 동료입니다. 하지만 **무엇이 맞는지 판단하고 책임지는 사람**은 여러분입니다. 수고 많았습니다.